# Submission — XGB + CB + DTW ensemble

Loads pre-trained XGBoost and CatBoost models (trained locally on full data with DTW features) and runs inference against the Kaggle competition test set.

**Inputs (attach as Kaggle datasets):**
- `rogii-wellbore-geology-prediction` (the competition data)
- A private dataset that bundles our `src/` + final model artefacts. Adjust `BUNDLE` below.

**Bundle layout expected:**
```
<bundle>/
  src/baseline.py
  artefacts/features.json
  artefacts/best_iters.json
  artefacts/ensemble_weights.json
  artefacts/final_xgb.json
  artefacts/final_cb.cbm
```

**Local OOF reference:** ensemble RMSE 12.531 on tag-grouped 5-fold CV (xgb=12.612, cb=12.661, NM weights cb=0.44 xgb=0.56).

In [ ]:
import os, sys, shutil
from pathlib import Path

# ── Paths — edit these to match your dataset slugs ───────────────────────
BUNDLE = Path('/kaggle/input/wellbore-baseline-bundle')
COMP   = Path('/kaggle/input/rogii-wellbore-geology-prediction')
WORK   = Path('/kaggle/working')

assert BUNDLE.exists(), f'Missing bundle dataset: {BUNDLE}'
assert COMP.exists(),   f'Missing competition input: {COMP}'

# Stage artefacts to /kaggle/working so baseline.py's ArtifactManager can
# write to it (Kaggle inputs are read-only).
ART_SRC = BUNDLE / 'artefacts'
ART_DST = WORK   / 'artefacts'
if not ART_DST.exists():
    shutil.copytree(ART_SRC, ART_DST)

# Tell baseline.py where to read/write before we import it.
os.environ['ARTEFACT_DIR'] = str(ART_DST)
os.environ['OUTPUT_DIR']   = str(WORK / 'outputs')
os.environ['DATA_DIR']     = str(COMP)

sys.path.insert(0, str(BUNDLE / 'src'))
print('Artefacts:', ART_DST)
print('Data:    ', COMP)
for p in sorted((ART_DST).iterdir()):
    print(' ', p.name)

In [ ]:
import baseline as bl
import numpy as np, pandas as pd

print('TRAIN_DIR :', bl.TRAIN_DIR)
print('TEST_DIR  :', bl.TEST_DIR, '(exists:', bl.TEST_DIR.exists(), ')')
print('ACTIVE_MODELS:', bl.ACTIVE_MODELS)
n_test = len(list(bl.TEST_DIR.glob('*__horizontal_well.csv')))
print(f'test wells: {n_test}')

## Build test features

Runs the same feature pipeline as training: GR rolling/lag/lead stats, geometry, beam search × 4, PF-Z, PF-ANCC, banded DTW. Single-pass over ~200 wells; expect ~3–5 minutes.

In [ ]:
test_df = bl.build_dataset(bl.TEST_DIR, is_train=False)
feature_cols = bl.am.load_json(bl.am.FEATURES_LIST)
missing = [c for c in feature_cols if c not in test_df.columns]
extra   = [c for c in test_df.columns if c not in feature_cols and c not in bl._META_COLS]
print(f'test_df: {test_df.shape}  features: {len(feature_cols)}  '
      f'missing: {len(missing)}  extra: {len(extra)}')
if missing:
    raise RuntimeError(f'test_df is missing {len(missing)} feature columns: {missing[:10]}')
test_df.to_parquet(WORK / 'test_df.parquet', index=False)

## Load models, ensemble, predict

In [ ]:
weights = bl.am.load_json(bl.am.ENSEMBLE_WEIGHTS)
models  = {k: bl.load_model(k, ART_DST) for k in bl.ACTIVE_MODELS}
print('weights:', weights)
print('models :', {k: type(m).__name__ for k, m in models.items()})

In [ ]:
# Per-model delta predictions
X = test_df[feature_cols]
test_preds = {k: bl.model_predict(k, m, X) for k, m in models.items()}
for k, p in test_preds.items():
    print(f'  {k.upper():3s} delta range: {p.min():.3f}  {p.mean():+.3f}  {p.max():.3f}')

# Convex ensemble + add back last_known_tvt
ensemble_delta = bl.apply_ensemble(test_preds, weights)
abs_tvt = ensemble_delta + test_df['last_known_tvt'].to_numpy()
print(f'\nensemble delta range: {ensemble_delta.min():.3f}  {ensemble_delta.mean():+.3f}  {ensemble_delta.max():.3f}')
print(f'abs TVT  range:       {abs_tvt.min():.3f}  {abs_tvt.mean():+.3f}  {abs_tvt.max():.3f}')

## Build submission

In [ ]:
sample = pd.read_csv(COMP / 'sample_submission.csv')
out    = WORK / 'submission.csv'
sub    = bl.build_submission(test_df, abs_tvt, sample, out)
print(sub.head())
print(f'\nrows: {len(sub):,}  rows-without-prediction: {sub["tvt"].isna().sum()}')
print(f'tvt summary:\n{sub["tvt"].describe()}')